# 04 Morphological Analysis

Computes the 17 extended morphological indices and compares the two groups.

**Inputs:** `kid_lit_100_ru.csv`, `kid_lit_100_foreign.csv`  
**Outputs:** `ru_morph_metric.csv`, `foreign_morph_metric.csv`, `morph_comparison.csv`

> **Not re-runnable from this repository.** This notebook reads the running text of the 100 books, which is under copyright and is not distributed. It is included as a record of how the released metrics were produced.


# Установка зависимостей

In [ ]:
!pip install razdel ruts --quiet
!python -m spacy download ru_core_news_sm --quiet

# Патчи и импорты

In [ ]:
import sys
import ast
import inspect
import math
import warnings
warnings.filterwarnings('ignore')

# Python 3.12 patch
if not hasattr(inspect, 'getargspec'):
    inspect.getargspec = lambda f: inspect.getfullargspec(f)[:4]

import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from scipy import stats
from statsmodels.stats.multitest import multipletests
from collections import Counter
from razdel import tokenize
import spacy
from ruts import MorphStats

sns.set_theme(style='whitegrid', palette='muted')
# Two-column ACL figure sizing + CB-safe palette
plt.rcParams.update({
    'font.family':        'DejaVu Sans',
    'font.size':          11,
    'axes.titlesize':     12,
    'axes.titleweight':   'bold',
    'axes.labelsize':     11,
    'xtick.labelsize':    10,
    'ytick.labelsize':    10,
    'legend.fontsize':    10,
    'figure.titlesize':   13,
    'figure.titleweight': 'bold',
    'savefig.dpi':        300,
    'figure.dpi':         120,
    'savefig.bbox':       'tight',
    'hatch.linewidth':    0.8,
    'patch.linewidth':    0.6,
})

# Wong (2011) colour-blind-safe palette, high B&W contrast
C_RU = '#0072B2'   # blue   – Russian (original)
C_FO = '#E69F00'   # orange – Russian (translated)
H_RU = '///'       # diagonal hatch – RU
H_FO = 'xxx'       # cross hatch    – FO

COL1 = 3.35        # single ACL column width (inches)
COL2 = 7.0         # full page width (inches)

nlp = spacy.load("ru_core_news_sm")
print("✅ OK!")

# Загрузка данных

In [ ]:
# --- Paths ---------------------------------------------------------------
# BASE_PATH must point at a directory holding this notebook's input files.
# Set the KIDLIT_BASE environment variable, or edit the fallback below.
# In Google Colab: mount Drive first, then set KIDLIT_BASE to the folder there.
import os
BASE_PATH = os.environ.get("KIDLIT_BASE", "../data/")


In [ ]:
USEFUL_COLS = [
    'id', 'title', 'original_title', 'author', 'author_gender',
    'year', 'language', 'country_of_origin', 'publisher', 'age_marker',
    'pages', 'illustrator', 'УДК', 'awards', 'is_translation',
    'translator', 'description', 'text', 'expert'
]

df_ru = pd.read_csv(
    BASE_PATH + 'kid_lit_100_ru.csv',
    sep=';', encoding='utf-8-sig',
    usecols=USEFUL_COLS
)

df_foreign = pd.read_csv(
    BASE_PATH + 'kid_lit_100_foreign.csv',
    sep=';', encoding='utf-8-sig',
    usecols=USEFUL_COLS
)

# Нормализация
for df in [df_ru, df_foreign]:
    df['age_marker']    = df['age_marker'].str.strip()
    df['author_gender'] = df['author_gender'].str.strip()

# Фильтруем книги без текста
df_ru_texts      = df_ru[df_ru['text'].notna()].copy().reset_index(drop=True)
df_foreign_texts = df_foreign[df_foreign['text'].notna()].copy().reset_index(drop=True)

print(f"📚 Русских книг всего:           {len(df_ru)}")
print(f"📝 Русских книг с текстом:       {len(df_ru_texts)}")
print(f"📚 Переводных книг всего:       {len(df_foreign)}")
print(f"📝 Переводных книг с текстом:   {len(df_foreign_texts)}")

# Функции токенизации и морфоанализа

In [ ]:
def tok_lst(txt):
    """Токенизация → список токенов"""
    if pd.isna(txt) or str(txt).strip() == '':
        return []
    return [_.text for _ in tokenize(str(txt))]

def tok_str(txt):
    """Токенизация → строка токенов (для ruts)"""
    if pd.isna(txt) or str(txt).strip() == '':
        return ''
    return ' '.join([_.text for _ in tokenize(str(txt))])

def morph_spacy(lst):
    """
    Морфоанализ через spacy → теги UD (pos=NOUN, case=Gen и т.д.)
    Принимает список или строку-список (после загрузки из CSV).
    """
    # Исправление: tok1_text после сохранения в CSV читается как строка
    if isinstance(lst, str):
        try:
            lst = ast.literal_eval(lst)
        except Exception:
            lst = lst.split()
    if not lst:
        return ''
    doc = nlp(' '.join(lst))
    result = []
    for token in doc:
        parts = [f"word={token.text}", f"pos={token.pos_}"]
        for feat, val in token.morph.to_dict().items():
            parts.append(f"{feat.lower()}={val}")
        result.append(' '.join(parts))
    return ' '.join(result)

def morph_ruts(txt):
    """Морфоанализ через ruts → теги pymorphy2 (PRTF, PRTS, PRED и т.д.)"""
    if pd.isna(txt) or str(txt).strip() == '':
        return ''
    ms = MorphStats(str(txt))
    return ' '.join([str(m) for m in ms.pos])

print("✅ Функции готовы")

#  Применение токенизации и морфоанализа

In [ ]:
for df, label, morph_path, metric_path in [
    (df_ru_texts,      'Русская',     MORPH_PATH + 'ru_morph.csv',
                                      MORPH_PATH + 'ru_morph_metric.csv'),
    (df_foreign_texts, 'Переводная', MORPH_PATH + 'foreign_morph.csv',
                                      MORPH_PATH + 'foreign_morph_metric.csv'),
]:
    print(f"⏳ [{label}] Токенизация...")
    df['tok1_text'] = df['text'].apply(tok_lst)
    df['tok2_text'] = df['text'].apply(tok_str)

    print(f"⏳ [{label}] Морфоанализ spacy...")
    df['morph1_text'] = df['tok1_text'].apply(morph_spacy)

    print(f"⏳ [{label}] Морфоанализ ruts...")
    df['morph2_text'] = df['tok2_text'].apply(morph_ruts)

    # Сохраняем tok1_text как строку, чтобы CSV читался корректно
    df['tok1_text'] = df['tok1_text'].apply(
        lambda x: ' '.join(x) if isinstance(x, list) else x
    )

    # quoting=csv.QUOTE_ALL — оборачивает все поля в кавычки,
    # благодаря чему переносы строк внутри текста не ломают таблицу
    import csv
    df.to_csv(morph_path, sep='\t', index=False,
              quoting=csv.QUOTE_ALL, escapechar='\\')
    print(f"✅ [{label}] Сохранено → {morph_path}")

print("\n✅ Морфоанализ завершён!")

# Морфологические метрики

In [ ]:
def function_word_metric(txt):
    """Индекс аналитичности/автосемантичности — доля служебных слов"""
    all_token = txt.count("word=")
    if all_token == 0: return 0
    return (txt.count("pos=ADP") + txt.count("pos=CCONJ") +
            txt.count("pos=SCONJ") + txt.count("pos=INTJ") +
            txt.count("pos=PART")) / all_token

def verb_word_metric(txt):
    """Индекс глагольности"""
    all_token = txt.count("word=")
    if all_token == 0: return 0
    return txt.count("pos=VERB") / all_token

def noun_word_metric(txt):
    """Индекс субстантивности"""
    all_token = txt.count("word=")
    if all_token == 0: return 0
    return (txt.count("pos=NOUN") + txt.count("pos=PROPN")) / all_token

def pronoun_word_metric(txt):
    """Индекс местоименности"""
    all_token = txt.count("word=")
    if all_token == 0: return 0
    return (txt.count("pos=PRON") + txt.count("pos=DET")) / all_token

def genitive_word_metric(txt):
    """Доля словоформ в родительном падеже"""
    all_token = txt.count("word=")
    if all_token == 0: return 0
    return txt.count("case=Gen") / all_token

def instrumental_word_metric(txt):
    """Доля словоформ в творительном падеже"""
    all_token = txt.count("word=")
    if all_token == 0: return 0
    return txt.count("case=Ins") / all_token

def short_adjective_word_metric(txt):
    """Доля кратких прилагательных"""
    all_token = txt.count("word=")
    if all_token == 0: return 0
    return txt.count("variant=Short") / all_token

def full_participle_word_metric(txt):
    """Доля полных причастий (ruts)"""
    all_token = len(txt.split())
    if all_token == 0: return 0
    return txt.count("PRTF") / all_token

def short_participle_word_metric(txt):
    """Доля кратких причастий (ruts)"""
    all_token = len(txt.split())
    if all_token == 0: return 0
    return txt.count("PRTS") / all_token

def predicative_word_metric(txt):
    """Доля предикативов (ruts)"""
    all_token = len(txt.split())
    if all_token == 0: return 0
    return txt.count("PRED") / all_token

def gerund_word_metric(txt):
    """Доля деепричастий"""
    all_token = txt.count("word=")
    if all_token == 0: return 0
    return txt.count("verbform=Conv") / all_token

def infinitive_word_metric(txt):
    """Доля инфинитивов"""
    all_token = txt.count("word=")
    if all_token == 0: return 0
    return txt.count("verbform=Inf") / all_token

def numeral_word_metric(txt):
    """Доля числительных"""
    all_token = txt.count("word=")
    if all_token == 0: return 0
    return txt.count("pos=NUM") / all_token

def particle_word_metric(txt):
    """Доля частиц"""
    all_token = txt.count("word=")
    if all_token == 0: return 0
    return txt.count("pos=PART") / all_token

def nominal_vocab_word_metric(txt):
    """Индекс именной лексики"""
    all_token = txt.count("word=")
    if all_token == 0: return 0
    return (txt.count("pos=NOUN") + txt.count("pos=PROPN") +
            txt.count("pos=ADJ")) / all_token

def noun_to_verb_word_metric(txt):
    """Соотношение имённости-глагольности"""
    verb_token = txt.count("pos=VERB")
    if verb_token == 0: return None
    return (txt.count("pos=NOUN") + txt.count("pos=PROPN")) / verb_token

def adjective_word_metric(txt1, txt2):
    """Индекс адъективности (прилагательные без причастий)"""
    all_token = txt1.count("word=")
    if all_token == 0: return 0
    return (txt1.count("pos=ADJ") -
            txt2.count("PRTF") - txt2.count("PRTS")) / all_token

print("✅ Метрики определены")

# Вычисление метрик

In [ ]:
def apply_metrics(df):
    """Применяет все морфологические метрики к датафрейму"""
    print("   Индекс аналитичности...")
    df['Индекс аналитичности/автосемантичности'] = df['morph1_text'].apply(function_word_metric)
    df['Индекс глагольности']                    = df['morph1_text'].apply(verb_word_metric)
    df['Индекс субстантивности']                 = df['morph1_text'].apply(noun_word_metric)
    df['Индекс местоименности']                  = df['morph1_text'].apply(pronoun_word_metric)
    df['Доля словоформ в родительном падеже']    = df['morph1_text'].apply(genitive_word_metric)
    df['Доля словоформ в творительном падеже']   = df['morph1_text'].apply(instrumental_word_metric)
    df['Доля кратких прилагательных']            = df['morph1_text'].apply(short_adjective_word_metric)
    df['Доля полных причастий']                  = df['morph2_text'].apply(full_participle_word_metric)
    df['Доля кратких причастий']                 = df['morph2_text'].apply(short_participle_word_metric)
    df['Доля предикативов']                      = df['morph2_text'].apply(predicative_word_metric)
    df['Доля деепричастий']                      = df['morph1_text'].apply(gerund_word_metric)
    df['Доля инфинитивов']                       = df['morph1_text'].apply(infinitive_word_metric)
    df['Доля числительных']                      = df['morph1_text'].apply(numeral_word_metric)
    df['Доля частиц']                            = df['morph1_text'].apply(particle_word_metric)
    df['Индекс именной лексики']                 = df['morph1_text'].apply(nominal_vocab_word_metric)
    df['Соотношение имённости-глагольности']     = df['morph1_text'].apply(noun_to_verb_word_metric)
    df['Индекс адъективности']                   = df.apply(
        lambda row: adjective_word_metric(row['morph1_text'], row['morph2_text']), axis=1
    )
    return df

print("⏳ Метрики для русской литературы...")
df_ru_texts = apply_metrics(df_ru_texts)
df_ru_texts.to_csv(MORPH_PATH + 'ru_morph_metric.csv', sep='\t', index=False)
print("✅ Сохранено → ru_morph_metric.csv")

print("⏳ Метрики для Переводной литературы...")
df_foreign_texts = apply_metrics(df_foreign_texts)
df_foreign_texts.to_csv(MORPH_PATH + 'foreign_morph_metric.csv', sep='\t', index=False)
print("✅ Сохранено → foreign_morph_metric.csv")

# Превью метрик

In [ ]:
METRIC_COLS = [
    'title', 'author', 'age_marker',
    'Индекс аналитичности/автосемантичности',
    'Индекс глагольности', 'Индекс субстантивности',
    'Индекс местоименности', 'Индекс адъективности',
    'Индекс именной лексики', 'Соотношение имённости-глагольности'
]

print("📊 РУССКАЯ ЛИТЕРАТУРА:")
display(df_ru_texts[METRIC_COLS])

print("\n📊 Переводная ЛИТЕРАТУРА:")
display(df_foreign_texts[METRIC_COLS])

# Сравнительный анализ

## Подготовка к сравнению

In [ ]:
df_ru_texts['group']      = 'Russian (original)'
df_foreign_texts['group'] = 'Russian (translated)'
df_all = pd.concat([df_ru_texts, df_foreign_texts], ignore_index=True)

MORPH_METRICS = {
    'Индекс аналитичности/автосемантичности': 'Function words',
    'Индекс глагольности':                    'Verbs',
    'Индекс субстантивности':                 'Nouns',
    'Индекс местоименности':                  'Pronouns',
    'Индекс адъективности':                   'Adjectives',
    'Индекс именной лексики':                 'Nominal vocab',
    'Соотношение имённости-глагольности':     'Noun/Verb ratio',
    'Доля словоформ в родительном падеже':    'Genitive case',
    'Доля словоформ в творительном падеже':   'Instrumental case',
    'Доля кратких прилагательных':            'Short adj.',
    'Доля полных причастий':                  'Full participles',
    'Доля кратких причастий':                 'Short participles',
    'Доля предикативов':                      'Predicatives',
    'Доля деепричастий':                      'Gerunds',
    'Доля инфинитивов':                       'Infinitives',
    'Доля числительных':                      'Numerals',
    'Доля частиц':                            'Particles',
}

# Период издания: до 2020 / с 2020
def get_period(y):
    return 'before 2020' if pd.notna(y) and int(y) < 2020 else '2020 and later'

df_all['period']             = df_all['year'].apply(get_period)
df_ru_texts['period']        = df_ru_texts['year'].apply(get_period)
df_foreign_texts['period']   = df_foreign_texts['year'].apply(get_period)

# Пол автора (жен/муж → коллектив)
def clean_gender(g):
    return 'collective' if str(g).strip() == 'жен/муж' else ('female' if str(g).strip() == 'жен' else ('male' if str(g).strip() == 'муж' else str(g).strip()))

df_all['gender_clean']           = df_all['author_gender'].apply(clean_gender)
df_ru_texts['gender_clean']      = df_ru_texts['author_gender'].apply(clean_gender)
df_foreign_texts['gender_clean'] = df_foreign_texts['author_gender'].apply(clean_gender)


# ── Accessibility constants (shared by all plot cells) ────────────────────
PAL_MORPH   = {'Russian (original)': C_RU, 'Russian (translated)': C_FO}
HATCH_MORPH = {'Russian (original)': H_RU, 'Russian (translated)': H_FO}
ORDER_MORPH = ['Russian (original)', 'Russian (translated)']
LEGEND_MORPH = [
    mpatches.Patch(facecolor=C_RU, hatch=H_RU, edgecolor='black',
                   label='Russian (original)'),
    mpatches.Patch(facecolor=C_FO, hatch=H_FO, edgecolor='black',
                   label='Russian (translated)'),
]
print(f"✅ Объединённый датафрейм: {len(df_all)} книг")
print(f"\nГруппировки для анализа:")
print(f"  group:         {df_all['group'].value_counts().to_dict()}")
print(f"  gender_clean:  {df_all['gender_clean'].value_counts().to_dict()}")
print(f"  language (ин): {df_foreign_texts['language'].value_counts().to_dict()}")
print(f"  period:        {df_all['period'].value_counts().to_dict()}")

## Сводная таблица + статистика (t-тест, Коэн d)

In [ ]:
print("\n" + "═"*70)
print("   СРАВНЕНИЕ МОРФОЛОГИЧЕСКИХ МЕТРИК: РУССКАЯ vs ПЕРЕВОДНАЯ")
print("═"*70)

comparison_rows = []
for metric, short_name in MORPH_METRICS.items():
    if metric not in df_all.columns:
        continue
    ru_vals = df_ru_texts[metric].dropna()
    fo_vals = df_foreign_texts[metric].dropna()
    if len(ru_vals) < 2 or len(fo_vals) < 2:
        continue

    t_stat, p_val   = stats.ttest_ind(ru_vals, fo_vals, equal_var=False)
    pooled_std      = np.sqrt((ru_vals.std()**2 + fo_vals.std()**2) / 2)
    cohen_d         = (ru_vals.mean() - fo_vals.mean()) / pooled_std \
                      if pooled_std > 0 else 0
    significance    = '***' if p_val < 0.001 else \
                      '**'  if p_val < 0.01  else \
                      '*'   if p_val < 0.05  else ''

    comparison_rows.append({
        'Метрика':       metric,
        'Кратко':        short_name,
        'Рус. среднее':  round(ru_vals.mean(), 4),
        'Ин. среднее':   round(fo_vals.mean(), 4),
        'Разница':       round(ru_vals.mean() - fo_vals.mean(), 4),
        'Рус. std':      round(ru_vals.std(), 4),
        'Ин. std':       round(fo_vals.std(), 4),
        't-статистика':  round(t_stat, 3),
        'p-значение':    round(p_val, 4),
        'Знч.':          significance,
        'Коэн d':        round(cohen_d, 3),
        'Выше у':        'Русская' if ru_vals.mean() > fo_vals.mean()
                         else 'Переводная',
    })

comp_df = pd.DataFrame(comparison_rows)

# ── Поправка на множественные сравнения (Benjamini–Hochberg FDR) ──────────
from statsmodels.stats.multitest import multipletests

_, p_adj, _, _ = multipletests(comp_df['p-значение'], alpha=0.05, method='fdr_bh')
comp_df['p-adj (FDR)'] = p_adj.round(4)
comp_df['Знч. adj'] = np.where(p_adj < 0.001, '***',
                      np.where(p_adj < 0.01,  '**',
                      np.where(p_adj < 0.05,  '*',  '')))

comp_df_sorted = comp_df.sort_values('p-значение')

print(comp_df_sorted[['Кратко', 'Рус. среднее', 'Ин. среднее',
                       'Разница', 'p-значение', 'Знч.', 'p-adj (FDR)', 'Знч. adj',
                       'Коэн d', 'Выше у']
                     ].to_string(index=False))
print("\n* p<0.05  ** p<0.01  *** p<0.001  (adj = FDR-corrected Benjamini–Hochberg)")

comp_df.to_csv(
    MORPH_PATH + 'morph_comparison.csv',
    sep='\t', index=False, encoding='utf-8-sig'
)
print("\n✅ Сохранено → morph_comparison.csv")

## График 1 — Grouped bar chart всех метрик

In [ ]:
metrics_list = [m for m in MORPH_METRICS if m in df_all.columns]
short_list   = [MORPH_METRICS[m] for m in metrics_list]

ru_means = [df_ru_texts[m].mean()      for m in metrics_list]
fo_means = [df_foreign_texts[m].mean() for m in metrics_list]
ru_stds  = [df_ru_texts[m].std()       for m in metrics_list]
fo_stds  = [df_foreign_texts[m].std()  for m in metrics_list]

x     = np.arange(len(metrics_list))
width = 0.38

fig, ax = plt.subplots(figsize=(COL2, 3.6))

bars_ru = ax.bar(x - width/2, ru_means, width, yerr=ru_stds,
                 label='Russian (original)', color=C_RU, hatch=H_RU,
                 edgecolor='black', error_kw=dict(ecolor='#333', capsize=3),
                 alpha=0.9)
bars_fo = ax.bar(x + width/2, fo_means, width, yerr=fo_stds,
                 label='Russian (translated)', color=C_FO, hatch=H_FO,
                 edgecolor='black', error_kw=dict(ecolor='#333', capsize=3),
                 alpha=0.9)

for i, metric in enumerate(metrics_list):
    row = comp_df[comp_df['Метрика'] == metric]
    if not row.empty and row['Знч. adj'].values[0]:
        y_pos = max(ru_means[i] + ru_stds[i],
                    fo_means[i] + fo_stds[i]) + 0.006
        ax.text(i, y_pos, row['Знч. adj'].values[0],
                ha='center', va='bottom', fontsize=9,
                color='#CC0000', fontweight='bold')

ax.set_xticks(x)
ax.set_xticklabels(short_list, rotation=38, ha='right', fontsize=9)
ax.set_title(
    'Morphological metrics: Russian (original) vs Russian (translated)\n'
    'whiskers = SD · asterisks = FDR-corrected (BH)',
    pad=6
)
ax.set_ylabel('Mean value (share)')
ax.legend(handles=LEGEND_MORPH, frameon=False)
ax.yaxis.grid(True, alpha=0.35)
ax.set_axisbelow(True)

plt.tight_layout()
plt.savefig(MORPH_PATH + 'plot_morph_bar.png', dpi=300, bbox_inches='tight')
plt.show()
print("✅ plot_morph_bar.png")

## График 2 — Boxplot ключевых метрик

In [ ]:
key_metrics = [
    'Индекс глагольности',
    'Индекс субстантивности',
    'Индекс местоименности',
    'Индекс адъективности',
    'Индекс именной лексики',
    'Доля деепричастий',
    'Доля инфинитивов',
    'Доля частиц',
    'Доля словоформ в родительном падеже',
    'Доля полных причастий',
    'Соотношение имённости-глагольности',
    'Индекс аналитичности/автосемантичности',
]
key_metrics = [m for m in key_metrics if m in df_all.columns]

n_cols = 4
n_rows = math.ceil(len(key_metrics) / n_cols)

fig, axes = plt.subplots(n_rows, n_cols,
                         figsize=(COL2, n_rows * 2.4))
fig.suptitle('Morphological metrics — distributions by group\n'
             'asterisks = FDR-corrected (BH)',
             y=1.01)

for i, metric in enumerate(key_metrics):
    ax = axes.flat[i]
    bp = sns.boxplot(data=df_all, x='group', y=metric, ax=ax,
                     palette=PAL_MORPH, order=ORDER_MORPH,
                     width=0.5, linewidth=0.9, fliersize=2.5)
    # Apply hatching
    patches = [p for p in ax.patches
               if isinstance(p, matplotlib.patches.PathPatch)]
    for p, grp in zip(patches, ORDER_MORPH):
        p.set_hatch(HATCH_MORPH[grp])
        p.set_edgecolor('black')
    # Individual points
    sns.stripplot(data=df_all, x='group', y=metric, ax=ax,
                  color='black', order=ORDER_MORPH,
                  size=2.2, alpha=0.35, jitter=True)
    row = comp_df[comp_df['Метрика'] == metric]
    if not row.empty:
        sign      = row['Знч. adj'].values[0]
        p_adj_val = row['p-adj (FDR)'].values[0]
        label_str = f"p_adj={p_adj_val:.3f}" + (f" {sign}" if sign else "")
        col_t     = '#CC0000' if sign else '#444444'
        ax.set_title(f"{MORPH_METRICS[metric]}\n{label_str}",
                     fontsize=9, color=col_t)
    else:
        ax.set_title(MORPH_METRICS.get(metric, metric), fontsize=9)
    ax.set_xlabel('')
    ax.set_ylabel('')
    ax.set_xticklabels(['Orig.', 'Transl.'], fontsize=8)

for j in range(len(key_metrics), len(axes.flat)):
    axes.flat[j].set_visible(False)

fig.legend(handles=LEGEND_MORPH, loc='upper center',
           bbox_to_anchor=(0.5, -0.02), ncol=2, frameon=False)
plt.tight_layout()
plt.savefig(MORPH_PATH + 'plot_morph_boxplot.png', dpi=300, bbox_inches='tight')
plt.show()
print("✅ plot_morph_boxplot.png")

## График 4 — Forest plot размеров эффектов (Коэн d)

In [ ]:
forest_df = comp_df.copy()
forest_df['Кратко'] = forest_df['Метрика'].map(MORPH_METRICS)
forest_df = forest_df.sort_values('Коэн d')

colors_f = [C_FO if d < 0 else C_RU for d in forest_df['Коэн d']]
hatch_f  = [H_FO  if d < 0 else H_RU  for d in forest_df['Коэн d']]

fig, ax = plt.subplots(figsize=(COL2, 4.8))

bars = ax.barh(forest_df['Кратко'], forest_df['Коэн d'],
               color=colors_f, edgecolor='black', alpha=0.9, height=0.6)
for bar, h in zip(bars, hatch_f):
    bar.set_hatch(h)

ax.axvline(0,     color='black', lw=1.0)
for v, ls in [(0.2, '--'), (-0.2, '--'), (0.5, ':'), (-0.5, ':')]:
    ax.axvline(v, color='#888', lw=0.8, linestyle=ls, alpha=0.6)

for i, (_, row) in enumerate(forest_df.iterrows()):
    if row['Знч. adj']:
        offset = 0.015 if row['Коэн d'] >= 0 else -0.015
        ha     = 'left' if row['Коэн d'] >= 0 else 'right'
        ax.text(row['Коэн d'] + offset, i, row['Знч. adj'],
                va='center', ha=ha, fontsize=9,
                color='#CC0000', fontweight='bold')

ax.set_xlabel(
    "Cohen's d   (+ = higher in original, − = higher in translated)\n"
    "dashed = small (0.2) · dotted = medium (0.5) · asterisks = FDR-corrected"
)
ax.set_title('Effect sizes: Russian (original) vs Russian (translated)')
ax.legend(handles=LEGEND_MORPH, loc='lower right', frameon=False)
ax.xaxis.grid(True, alpha=0.35)
ax.set_axisbelow(True)
ax.tick_params(axis='y', labelsize=9)

plt.tight_layout()
plt.savefig(MORPH_PATH + 'plot_morph_effect_size.png', dpi=300, bbox_inches='tight')
plt.show()
print("✅ plot_morph_effect_size.png")


## График 5 — Heatmap корреляций внутри каждой группы

In [ ]:
heatmap_metrics = [m for m in MORPH_METRICS
                   if m in df_all.columns
                   and m != 'Соотношение имённости-глагольности']

fig, axes = plt.subplots(1, 2, figsize=(COL2, 3.8))
fig.suptitle('Correlations of morphological metrics within groups', y=1.01)

for ax, grp_df, title in zip(
        axes,
        [df_ru_texts, df_foreign_texts],
        ['Russian (original)', 'Russian (translated)']):
    corr = grp_df[heatmap_metrics].corr()
    short_names  = [MORPH_METRICS[m] for m in heatmap_metrics]
    corr.index   = short_names
    corr.columns = short_names
    sns.heatmap(
        corr, ax=ax,
        cmap='RdBu_r', center=0, vmin=-1, vmax=1,
        annot=True, fmt='.2f', linewidths=0.4,
        annot_kws={'size': 7}, square=True,
        mask=np.triu(np.ones_like(corr, dtype=bool)),
        cbar_kws={'shrink': 0.7}
    )
    ax.set_title(title, pad=4)
    ax.tick_params(axis='x', rotation=40, labelsize=7)
    ax.tick_params(axis='y', labelsize=7)

plt.tight_layout()
plt.savefig(MORPH_PATH + 'plot_morph_heatmap.png', dpi=300, bbox_inches='tight')
plt.show()
print("✅ plot_morph_heatmap.png")

## Итог сравнения

In [ ]:
# Значимые по скорректированным p-значениям (FDR)
sig     = comp_df[comp_df['Знч. adj'] != ''].sort_values('p-adj (FDR)')
sig_raw = comp_df[comp_df['Знч.'] != '']   # for reference — before correction
large   = comp_df[comp_df['Коэн d'].abs() >= 0.5].sort_values(
    'Коэн d', key=abs, ascending=False
)

print("\n" + "═"*60)
print("  MORPHOLOGICAL ANALYSIS — SUMMARY")
print("═"*60)
print(f"  Russian (original):                  {len(df_ru_texts)}")
print(f"  Russian (translated):                {len(df_foreign_texts)}")
print(f"  Metrics compared:                    {len(comp_df)}")
print(f"  Significant before correction (p<0.05): {len(sig_raw)}")
print(f"  Significant after FDR (p_adj<0.05):     {len(sig)}")
print("  (Benjamini–Hochberg correction, α=0.05)")

if not sig.empty:
    print("\n  📌 FDR-robust differences:")
    for _, r in sig.iterrows():
        print(f"     {r['Кратко']:30s} "
              f"p={r['p-значение']:.4f}  p_adj={r['p-adj (FDR)']:.4f}{r['Знч. adj']:4s} "
              f"d={r['Коэн d']:+.3f}  ({r['Выше у']})")
else:
    print("\n  ℹ️  No metrics survive FDR correction.")
    print("     Non-zero effects (see below) should be treated as tendencies.")

if not large.empty:
    print("\n  📌 Large effect size (|d| ≥ 0.5):")
    for _, r in large.iterrows():
        print(f"     {r['Кратко']:30s} d={r['Коэн d']:+.3f}  "
              f"p_adj={r['p-adj (FDR)']:.4f}  ({r['Выше у']})")

print("\n  Data files:")
for f in ['ru_morph.csv', 'ru_morph_metric.csv',
          'foreign_morph.csv', 'foreign_morph_metric.csv',
          'morph_comparison.csv']:
    print(f"     {f}")

print("\n  Figures:")
for p in ['plot_morph_bar', 'plot_morph_boxplot',
          'plot_morph_effect_size', 'plot_morph_heatmap']:
    print(f"     {p}.png")